Notebook 3 — Train / validation / test split
-------

In [18]:
import sqlite3
import pandas as pd
import os

In [19]:
# connect to db and showing available tables inside
db_path = r"D:\2026\MLOps-Qafza-2026\database\olist.db"

conn = sqlite3.connect(db_path)

query = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

tables = pd.read_sql_query(query, conn)
display(tables)

,name
0,agg_olist_order_items
1,agg_olist_order_payments
2,agg_olist_order_reviews
3,ml_orders
4,ml_orders_labled
5,olist_customers
6,olist_geolocation
7,olist_order_items
8,olist_order_payments
9,olist_order_reviews


In [20]:
# read and view the table "ml_orders_labled"
query = '''
select * from ml_orders_labled
'''

ml_orders_labled = pd.read_sql_query(query, conn)
display(ml_orders_labled)
print('the shape of ml_orders_labled: ', ml_orders_labled.shape)



,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,number_of_items,total_freight_value,total_price,number_of_sellers,number_of_products,number_of_payments,total_payment_value,number_of_payment_types,max_payment_installments,is_late_label
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,8.72,29.99,1.0,1.0,3.0,38.71,2.0,1.0,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,22.76,118.70,1.0,1.0,1.0,141.46,1.0,1.0,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,19.22,159.90,1.0,1.0,1.0,179.12,1.0,3.0,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,27.20,45.00,1.0,1.0,1.0,72.20,1.0,1.0,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,8.72,19.90,1.0,1.0,1.0,28.62,1.0,1.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96471,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00,1.0,13.08,72.00,1.0,1.0,1.0,85.08,1.0,3.0,0
96472,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00,1.0,20.10,174.90,1.0,1.0,1.0,195.00,1.0,3.0,0
96473,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00,1.0,65.02,205.99,1.0,1.0,1.0,271.01,1.0,5.0,0
96474,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00,2.0,81.18,359.98,1.0,1.0,1.0,441.16,1.0,4.0,0


the shape of ml_orders_labled:  (96476, 18)


| Question                        | Random split  | Time-based split |
| ------------------------------- | ------------- | ---------------- |
| Data is randomly distributed    | Yes           | No               |
| Keeps chronological order       | No            | Yes              |
| Can use `stratify`              | Yes           | Not normally     |
| Keeps label ratio similar       | Yes           | Not necessarily  |
| Simulates predicting the future | Less directly | Yes              |
| Good choice when time matters   | Usually no    | Yes              |


time-based split
------

In [21]:
# Check the available date range
# inspect data, Before deciding between a random split and a time-based split

date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
# convert to datetime
for column in date_columns:
    ml_orders_labled[column] = pd.to_datetime(
    ml_orders_labled[column])

ml_orders_labled[date_columns].agg(["min", "max"])


,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date
min,2016-09-15 12:16:38,2016-10-11 13:46:32,2016-10-04
max,2018-08-29 15:00:37,2018-10-17 13:22:46,2018-10-25


In [22]:
# Does the target distribution change over time?
# Check the late-delivery rate by month:
# For a binary target encoded as 0 and 1, the mean equals the percentage/proportion of rows belonging to class 1.
monthly_late_rate = (
    ml_orders_labled
    .groupby(
        ml_orders_labled["order_purchase_timestamp"].dt.to_period("M")
    )["is_late_label"]
    .agg(["count", "mean"])
)
# count → number of orders in each month
# mean → late-delivery rate in each month
monthly_late_rate

,count,mean
order_purchase_timestamp,,
2016-09,1,1.000000
2016-10,270,0.011111
2016-12,1,0.000000
2017-01,750,0.030667
2017-02,1653,0.032063
2017-03,2546,0.055774
2017-04,2303,0.078593
2017-05,3545,0.036107
2017-06,3135,0.038596


In [23]:
# split & check the date range and the label balance in each split
ml_orders_labled = ml_orders_labled.sort_values(
    "order_purchase_timestamp"
).reset_index(drop=True)

n = len(ml_orders_labled)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

# split & creating the dfs
time_base_train = ml_orders_labled.iloc[:train_end]
time_base_validation = ml_orders_labled.iloc[train_end:validation_end]
time_base_test = ml_orders_labled.iloc[validation_end:]
# ===========================================================

for name, df in {
    "time_base_Train": time_base_train,
    "time_base_Validation": time_base_validation,
    "time_base_Test": time_base_test
}.items():
    
    print(f"\n{name}")
    print("number of Rows:", len(df))
    print(
        "Date range:",
        df["order_purchase_timestamp"].min(),
        "to",
        df["order_purchase_timestamp"].max()
    )
    print("Late rate:", df["is_late_label"].mean())


time_base_Train
number of Rows: 67533
Date range: 2016-09-15 12:16:38 to 2018-04-15 20:07:56
Late rate: 0.0902817881628241

time_base_Validation
number of Rows: 14471
Date range: 2018-04-15 20:10:23 to 2018-06-21 07:50:39
Late rate: 0.053417179185958126

time_base_Test
number of Rows: 14472
Date range: 2018-06-21 08:29:29 to 2018-08-29 15:00:37
Late rate: 0.06612769485903815


random-base split
-------------

In [24]:
# Create Train and Temporary Split
# Train = 70%  ,  Temp  = 30%

from sklearn.model_selection import train_test_split

train, temp = train_test_split(
    ml_orders_labled,
    test_size=0.30,
    random_state=42,  # to get the same split every run, making experiments reproducible.
    stratify=ml_orders_labled["is_late_label"] # to make sure the distribution of target classes stays approximately the same ration in each set.
)

# train_test_split function parameters:
# Parameter	            Description
# X	                    Feature data (input variables)
# y	                    Target data (output labels)
# test_size	            Fraction of data for testing (e.g., 0.2 = 20%)
# train_size	        Fraction of data for training (optional)
# random_state	        Controls randomness for reproducible results
# shuffle	            Whether to shuffle data before splitting (default=True)
# stratify	            Preserves class distribution in classification problems


In [27]:
# Split Temporary into Validation and Test
# Validation = 15%    ,	Test = 15%
validation, test = train_test_split(
    temp,
    test_size=0.50,
    random_state=42,
    stratify=temp["is_late_label"]
)



In [28]:
# Check the sizes
print(f"Train rows:      {len(train):,}")
print(f"Validation rows: {len(validation):,}")
print(f"Test rows:       {len(test):,}")
print('=' * 25)
print(f"Train percent:      {len(train) / len(ml_orders_labled):.2%}")
print(f"Validation percent: {len(validation) / len(ml_orders_labled):.2%}")
print(f"Test percent:       {len(test) / len(ml_orders_labled):.2%}")


Train rows:      67,533
Validation rows: 14,471
Test rows:       14,472
Train percent:      70.00%
Validation percent: 15.00%
Test percent:       15.00%


In [29]:
# Check the label balance
# should see approximately the same ratio because of using stratify
for name, df in {
    "Train": train,
    "Validation": validation,
    "Test": test
}.items():

    print(f"\n{name}")
    print(df["is_late_label"].value_counts(normalize=True))



Train
is_late_label
0    0.918869
1    0.081131
Name: proportion, dtype: float64

Validation
is_late_label
0    0.918872
1    0.081128
Name: proportion, dtype: float64

Test
is_late_label
0    0.918878
1    0.081122
Name: proportion, dtype: float64


In [30]:
# showing the label balance as table format
split_distribution = pd.DataFrame({
    "Train": train["is_late_label"].value_counts(normalize=True),
    "Validation": validation["is_late_label"].value_counts(normalize=True),
    "Test": test["is_late_label"].value_counts(normalize=True)
})

split_distribution.index = split_distribution.index.map({
    0: "On Time",
    1: "Late"
})

split_distribution


,Train,Validation,Test
is_late_label,,,
On Time,0.918869,0.918872,0.918878
Late,0.081131,0.081128,0.081122


In [31]:
# Check the date range in each split

for name, df in {
    "Train": train,
    "Validation": validation,
    "Test": test
}.items():

    print(
        f"{name}:",
        df["order_purchase_timestamp"].min(),
        "→",
        df["order_purchase_timestamp"].max()
    )


Train: 2016-09-15 12:16:38 → 2018-08-29 15:00:37
Validation: 2016-10-03 22:31:31 → 2018-08-29 14:18:28
Test: 2016-10-03 21:01:41 → 2018-08-29 14:52:00


In [33]:
# Check for overlap between splits
# Since each order should appear in only one dataset, should verify this:
# This is a useful data-leakage check.
print(
    "Train in Validation:",
    len(set(train["order_id"]) & set(validation["order_id"]))
)

print(
    "Train in Test:",
    len(set(train["order_id"]) & set(test["order_id"]))
)

print(
    "Validation in Test:",
    len(set(validation["order_id"]) & set(test["order_id"]))
)


Train in Validation: 0
Train in Test: 0
Validation in Test: 0


select and save the train, valifation , test tables
-----------

Decide between random or time-based splitting:  
will use random as need to keeps the same percentage of late and on-time orders in all three datasets (train, validate,test) and not care for the time dependency.


In [35]:
# Save the artifacts
train.to_sql('training_set', conn, if_exists="replace", index=False)
validation.to_sql('validation_set', conn, if_exists="replace", index=False)
test.to_sql('testing_set', conn, if_exists="replace", index=False)

print("Train, validation, and test tables created.")

Train, validation, and test tables created.


In [36]:
# display the names for all tables inside the db
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

tables = pd.read_sql_query(query, conn) 
tables

,name
0,agg_olist_order_items
1,agg_olist_order_payments
2,agg_olist_order_reviews
3,ml_orders
4,ml_orders_labled
5,olist_customers
6,olist_geolocation
7,olist_order_items
8,olist_order_payments
9,olist_order_reviews


In [ ]:
# display the training_set
query = 'select * from training_set'
tables = pd.read_sql_query(query, conn) 
tables

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,number_of_items,total_freight_value,total_price,number_of_sellers,number_of_products,number_of_payments,total_payment_value,number_of_payment_types,max_payment_installments,is_late_label
0,2bedc0d7231e504d0476c606108e7e73,fff93c1da78dafaaa304ff032abc6205,delivered,2018-06-13 01:57:22,2018-06-13 02:59:41,2018-06-18 14:01:00,2018-06-21 11:56:46,2018-07-11 00:00:00,3.0,43.60,198.89,1.0,2.0,1.0,242.48,1.0,10.0,0
1,202c632ebb14ebfabad48f43c9a9f166,d7263dd40ab3973760e65588299bfaa8,delivered,2018-05-14 12:55:16,2018-05-14 13:17:09,2018-05-14 15:12:00,2018-05-22 23:16:46,2018-05-28 00:00:00,1.0,15.41,189.90,1.0,1.0,1.0,205.31,1.0,8.0,0
2,df9deeb3f35f5f1f50697012b7bab129,22f376753863f196880ee0a11aead3e1,delivered,2018-01-13 13:36:23,2018-01-16 04:31:07,2018-01-19 01:58:40,2018-02-02 01:06:08,2018-03-02 00:00:00,1.0,58.54,107.00,1.0,1.0,1.0,165.54,1.0,1.0,0
3,1129997904eb031a05d781b6179078d7,f740e6e44d74fb1e93f1bcd947925230,delivered,2018-01-10 15:54:11,2018-01-10 16:09:53,2018-01-11 23:03:37,2018-01-12 22:37:36,2018-01-29 00:00:00,1.0,9.37,79.90,1.0,1.0,1.0,89.27,1.0,1.0,0
4,b8ee80bb6d575240f0b5f9f87987be6d,d3bc1fd607f5e867e30371619d28a55b,delivered,2018-06-11 20:22:26,2018-06-11 20:40:57,2018-06-13 13:37:00,2018-07-05 15:58:26,2018-06-29 00:00:00,1.0,68.45,265.00,1.0,1.0,1.0,333.45,1.0,10.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67528,6a1dff28da32815c94da401e19d2a7f8,29251d6a551cedc7040b152221744460,delivered,2017-03-21 11:33:31,2017-03-21 11:33:31,2017-03-27 14:13:02,2017-03-31 15:34:11,2017-04-13 00:00:00,1.0,10.96,30.00,1.0,1.0,1.0,40.96,1.0,1.0,0
67529,31874b1728b2dc2adbd7ac767892fce0,daf22b7353d8e5021df9894b32dd6b45,delivered,2018-06-02 08:47:12,2018-06-02 08:55:27,2018-06-04 13:49:00,2018-06-26 18:50:45,2018-07-17 00:00:00,1.0,51.13,89.90,1.0,1.0,1.0,141.03,1.0,4.0,0
67530,d8f92d7e898a1027a0957b82f58ddbc4,3278f0116538282732803d2b35a9fbd6,delivered,2018-08-21 20:10:39,2018-08-21 20:25:19,2018-08-22 12:27:00,2018-08-23 18:58:41,2018-08-24 00:00:00,1.0,7.65,56.00,1.0,1.0,1.0,63.65,1.0,1.0,0
67531,35011b1b3211ea7c58a699e4b8bfe93d,9c9da709db4d73a4b732e11026252c3c,delivered,2017-06-22 15:00:45,2017-06-22 15:10:20,2017-06-26 15:39:44,2017-07-03 16:54:58,2017-07-05 00:00:00,1.0,12.48,59.70,1.0,1.0,1.0,72.18,1.0,1.0,0


In [ ]:
# display the validation_set
query = 'select * from validation_set'
tables = pd.read_sql_query(query, conn) 
tables

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,number_of_items,total_freight_value,total_price,number_of_sellers,number_of_products,number_of_payments,total_payment_value,number_of_payment_types,max_payment_installments,is_late_label
0,c88921a6cbda2e9973f9cfbd4e771d20,dc29649b0a10811a1a7e96a23c537999,delivered,2017-03-24 15:02:23,2017-03-24 15:15:12,2017-03-28 08:33:19,2017-04-04 15:52:26,2017-04-12 00:00:00,2.0,17.44,25.98,1.0,2.0,1.0,43.42,1.0,4.0,0
1,a9c3f5f3721d134e94565756f8a60ef0,8ee32f5a8a3a795ac11b2100b4558dba,delivered,2018-04-20 15:31:00,2018-04-24 18:41:22,2018-04-25 13:38:00,2018-05-02 23:32:20,2018-05-22 00:00:00,1.0,18.23,29.99,1.0,1.0,1.0,48.22,1.0,1.0,0
2,0ad0daff42cf44b50d0bac4c64722e4f,19970def56bdc13635f5591fc47a3727,delivered,2017-04-30 11:27:15,2017-04-30 11:35:19,2017-05-05 13:58:06,2017-05-12 07:07:51,2017-05-25 00:00:00,2.0,33.12,386.25,1.0,2.0,1.0,419.37,1.0,10.0,0
3,a4d873bfcdebc189c6bfeaef27325df0,4a068788cb64b9a1e8a4fd2502ee19db,delivered,2017-12-06 22:13:04,2017-12-06 22:32:58,2017-12-08 22:09:48,2017-12-30 02:37:02,2018-01-05 00:00:00,1.0,26.73,85.00,1.0,1.0,1.0,111.73,1.0,1.0,0
4,36a99d1efdc9f0b6d7a7f793d0f304b0,f4ef757008c51dc9573991137fe68ca4,delivered,2018-08-19 09:44:58,2018-08-20 14:53:14,2018-08-21 14:46:00,2018-08-23 22:11:56,2018-08-30 00:00:00,1.0,12.79,12.00,1.0,1.0,1.0,24.79,1.0,1.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14466,d29d63928559ca53630a4e064a42381a,1c6a3211e5215fb2cbc2f392b76fb37b,delivered,2017-03-27 11:33:36,2017-03-28 06:42:28,2017-03-28 15:10:04,2017-04-10 13:41:45,2017-05-05 00:00:00,1.0,25.18,99.00,1.0,1.0,1.0,124.18,1.0,1.0,0
14467,d84d70e52f7b7abba77d84d49d97a134,873834ac00ab80de4d9cb2decaf0af61,delivered,2018-08-17 22:17:00,2018-08-17 22:30:13,2018-08-20 13:47:00,2018-08-27 18:32:08,2018-09-04 00:00:00,1.0,18.15,49.90,1.0,1.0,1.0,68.05,1.0,1.0,0
14468,ab459ccd2e24ef6d6390517c8b9809a4,fe510f4e4c3ea978c714418b91049d9e,delivered,2017-09-19 22:26:27,2017-09-21 02:45:51,2017-09-22 16:27:42,2017-10-02 22:45:44,2017-10-19 00:00:00,2.0,32.62,156.00,1.0,1.0,1.0,188.62,1.0,1.0,0
14469,aca3b492e82b71863328c4c288861ca2,9be914d71f87403b18d4008de773d816,delivered,2018-02-27 22:35:46,2018-02-27 22:48:15,2018-03-09 22:09:38,2018-03-14 21:06:57,2018-03-27 00:00:00,1.0,13.14,144.41,1.0,1.0,1.0,157.55,1.0,2.0,0


In [ ]:
# display the testing_set
query = 'select * from testing_set'
tables = pd.read_sql_query(query, conn) 
tables

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,number_of_items,total_freight_value,total_price,number_of_sellers,number_of_products,number_of_payments,total_payment_value,number_of_payment_types,max_payment_installments,is_late_label
0,2ecf1e81869aad99fb1eaf286a7eb119,e38e171f81900e949c42cf7901227c37,delivered,2017-12-10 17:51:35,2017-12-12 04:09:01,2017-12-15 12:49:15,2017-12-27 14:43:45,2018-01-08 00:00:00,1.0,15.10,24.99,1.0,1.0,1.0,40.09,1.0,1.0,0
1,164a6c920f4a6f51223792333af13e24,2fb547e1f38b78f5f2073b68ceee4f53,delivered,2018-01-13 01:18:04,2018-01-13 01:28:35,2018-01-15 13:48:20,2018-01-30 00:38:48,2018-02-16 00:00:00,1.0,16.11,44.90,1.0,1.0,1.0,63.12,1.0,2.0,0
2,b92eedda04f5afe987c7e439cd6d971b,77dc7a088265a3d7b01dfd9c1ff38e8e,delivered,2018-01-19 16:58:07,2018-01-20 09:10:35,2018-01-24 00:23:37,2018-02-08 12:28:01,2018-02-19 00:00:00,1.0,20.76,858.90,1.0,1.0,1.0,879.66,1.0,1.0,0
3,7e41633529000f80967da24083bb6c1c,451ac26d4d2da6b319318c13aac25937,delivered,2018-07-26 20:37:12,2018-07-26 21:10:17,2018-07-27 12:06:00,2018-08-02 20:59:41,2018-08-17 00:00:00,1.0,23.29,69.90,1.0,1.0,2.0,93.19,2.0,1.0,0
4,65dfd5eb6f5db48fcf9b5e4f5c8f1391,c3d1f7d1843910b559fe6ea1f7f09c7d,delivered,2017-11-01 14:10:46,2017-11-01 14:31:34,2017-11-03 17:13:50,2017-11-08 20:34:45,2017-11-29 00:00:00,1.0,21.72,175.00,1.0,1.0,1.0,196.72,1.0,1.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14467,a38aeb63e6290170677c4cf2ab4c0f7f,62e1860ef3884fd9e103e75783dca1c0,delivered,2018-03-26 10:14:31,2018-03-26 10:31:11,2018-03-26 22:12:20,2018-04-02 18:34:32,2018-04-17 00:00:00,1.0,18.89,144.90,1.0,1.0,1.0,163.79,1.0,6.0,0
14468,cea05d3139cd44136f17235b6f42969f,d999be4ad55ab611a78cbaffcee30aa9,delivered,2018-07-02 09:13:20,2018-07-05 16:14:44,2018-07-05 14:00:00,2018-07-10 19:03:38,2018-07-27 00:00:00,1.0,18.36,37.50,1.0,1.0,1.0,55.86,1.0,1.0,0
14469,8f3f26027859349e29776a53fcf17963,77678cf0733734587184d89b548c3517,delivered,2017-04-06 12:12:21,2017-04-06 13:05:36,2017-04-07 15:39:46,2017-04-13 07:39:05,2017-04-27 00:00:00,1.0,10.96,18.99,1.0,1.0,1.0,29.95,1.0,1.0,0
14470,21acf351fd39c8a80aa62aed0b2fd28a,34d1179b8485fd694eaeaf5ba3ec0664,delivered,2017-10-11 13:27:06,2017-10-14 18:34:58,2017-10-16 21:22:35,2017-10-23 22:39:44,2017-11-01 00:00:00,1.0,17.67,59.90,1.0,1.0,1.0,77.57,1.0,1.0,0
